# GTSRB Traffic-Sign Recognition — Standalone Colab GPU Notebook

This notebook is completely self-contained: it does not require the project ZIP, GitHub, Google Drive, or any local source code. It downloads the verified GTSRB archive directly into the Colab runtime, trains two models on the selected GPU, evaluates the official test split, creates graphs and Grad-CAM, and downloads a results ZIP directly to your computer.

Select **Runtime → Change runtime type → T4/L4 GPU** before running all cells from top to bottom.

In [ ]:
# Install only lightweight utilities; TensorFlow is supplied by Colab.
%pip -q install pandas matplotlib scikit-learn Pillow opencv-python-headless
import os, json, zipfile, hashlib, urllib.request, shutil, random
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
gpus=tf.config.list_physical_devices('GPU')
print('TensorFlow:',tf.__version__)
print('GPU devices:',gpus)
assert gpus, 'GPU not detected. Select a T4/L4 GPU runtime and rerun this cell.'
for gpu in gpus: tf.config.experimental.set_memory_growth(gpu, True)
print('GPU ready:',tf.config.experimental.get_device_details(gpus[0]).get('device_name','GPU'))

In [ ]:
# Direct Zenodo download, checksum verification, extraction, and optional browser download.
ROOT=Path('/content/gtsrb_colab')
ROOT.mkdir(exist_ok=True)
archive=ROOT/'data.zip'
url='https://zenodo.org/api/records/13741936/files/data.zip/content'
expected_md5='7faafd0d78d831c7eb4d555641a83a62'
if not archive.exists() or hashlib.md5(archive.read_bytes()).hexdigest()!=expected_md5:
    print('Downloading complete 320 MB GTSRB archive...')
    urllib.request.urlretrieve(url, archive)
actual_md5=hashlib.md5(archive.read_bytes()).hexdigest()
print('MD5:',actual_md5)
assert actual_md5==expected_md5, 'Checksum mismatch'
DATA=ROOT/'gtsrb'
if not (DATA/'Train').exists():
    with zipfile.ZipFile(archive) as z: z.extractall(DATA)
print('Dataset:',DATA)
print('Train classes:',len(list((DATA/'Train').glob('[0-9]*'))))
# Uncomment to download the verified 320 MB archive directly to your computer:
# from google.colab import files; files.download(str(archive))

In [ ]:
# Load the official training and test labels.
def group_id(path):
    parts=path.stem.split('_')
    return f'{path.parent.name}_{parts[1] if len(parts)>2 else parts[0]}'
train_rows=[]
for folder in sorted((DATA/'Train').iterdir()):
    if folder.is_dir() and folder.name.isdigit():
        for p in folder.iterdir():
            if p.suffix.lower() in {'.png','.ppm','.jpg','.jpeg'}: train_rows.append({'path':str(p),'label':int(folder.name),'group':group_id(p)})
train_df=pd.DataFrame(train_rows)
test_labels=pd.read_csv(DATA/'Test.csv')
test_df=pd.DataFrame({'path':[str(DATA/'Test'/Path(str(p)).name) for p in test_labels['Path']], 'label':test_labels['ClassId'].astype(int)})
test_df['group']=test_df['path'].map(lambda x:group_id(Path(x)))
assert set(train_df.label)==set(range(43)) and len(train_df)==39209 and len(test_df)==12630
print(f'train={len(train_df):,}; official_test={len(test_df):,}; classes={train_df.label.nunique()}')
counts=train_df.label.value_counts().sort_index()
plt.figure(figsize=(16,4)); plt.bar(counts.index,counts.values,color='#19c8ba'); plt.title('GTSRB training images by class'); plt.xlabel('Class ID'); plt.ylabel('Images'); plt.show()

In [ ]:
# Grouped validation split and tf.data image pipeline.
splitter=GroupShuffleSplit(n_splits=1,test_size=0.15,random_state=SEED)
tr_idx,val_idx=next(splitter.split(train_df,train_df.label,groups=train_df.group))
fit_df=train_df.iloc[tr_idx].reset_index(drop=True); val_df=train_df.iloc[val_idx].reset_index(drop=True)
def make_ds(frame,size,batch=128,shuffle=False):
    ds=tf.data.Dataset.from_tensor_slices((frame.path.values,frame.label.values.astype('int32')))
    def load(path,label):
        image=tf.io.decode_image(tf.io.read_file(path),channels=3,expand_animations=False)
        image=tf.image.resize(image,[size,size]); return tf.cast(image,tf.float32)/255.,label
    ds=ds.map(load,num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle: ds=ds.shuffle(min(len(frame),10000),seed=SEED,reshuffle_each_iteration=True)
    return ds.batch(batch).prefetch(tf.data.AUTOTUNE)
print(f'fit={len(fit_df):,}; validation={len(val_df):,}; grouped split ready')

In [ ]:
# Model definitions. Moderate augmentation is active only during training.
def aug(): return tf.keras.Sequential([tf.keras.layers.RandomRotation(.04),tf.keras.layers.RandomTranslation(.05,.05),tf.keras.layers.RandomZoom(.08),tf.keras.layers.RandomContrast(.10)],name='augmentation')
def custom_cnn():
    x_in=tf.keras.Input((64,64,3)); x=aug()(x_in)
    for f in (32,64,128):
        x=tf.keras.layers.Conv2D(f,3,padding='same',activation='relu')(x); x=tf.keras.layers.BatchNormalization()(x); x=tf.keras.layers.MaxPooling2D()(x); x=tf.keras.layers.Dropout(.2)(x)
    x=tf.keras.layers.GlobalAveragePooling2D()(x); x=tf.keras.layers.Dense(128,activation='relu')(x); x=tf.keras.layers.Dropout(.3)(x)
    return tf.keras.Model(x_in,tf.keras.layers.Dense(43,activation='softmax')(x),name='custom_cnn')
def mobilenet():
    x_in=tf.keras.Input((128,128,3)); x=aug()(x_in); x=tf.keras.layers.Rescaling(2,-1)(x)
    base=tf.keras.applications.MobileNetV2(include_top=False,weights='imagenet',input_shape=(128,128,3),pooling='avg',name='mobilenet_backbone'); base.trainable=False
    x=base(x,training=False); x=tf.keras.layers.Dropout(.25)(x)
    return tf.keras.Model(x_in,tf.keras.layers.Dense(43,activation='softmax')(x),name='mobilenet_v2')
print('Model builders ready')

In [ ]:
# Strong Colab GPU training: 30 epochs custom CNN and 25 epochs MobileNetV2.
EPOCHS={'custom_cnn':30,'mobilenet_v2':25}; BATCH=128; OUT=ROOT/'artifacts'; OUT.mkdir(exist_ok=True)
def run_model(name):
    size=64 if name=='custom_cnn' else 128
    model=custom_cnn() if name=='custom_cnn' else mobilenet()
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),loss='sparse_categorical_crossentropy',metrics=['accuracy'])
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_accuracy',patience=6,restore_best_weights=True),tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=.4,patience=3,min_lr=1e-6),tf.keras.callbacks.ModelCheckpoint(OUT/f'{name}_best.keras',monitor='val_accuracy',save_best_only=True)]
    hist=model.fit(make_ds(fit_df,size,BATCH,True),validation_data=make_ds(val_df,size,BATCH),epochs=EPOCHS[name],callbacks=callbacks,verbose=1)
    model.save(OUT/f'{name}.keras')
    return model,hist.history,size
models={}; histories={}; sizes={}
for name in ('custom_cnn','mobilenet_v2'):
    models[name],histories[name],sizes[name]=run_model(name)

In [ ]:
# Official test evaluation, report files, confusion matrices, and learning curves.
def evaluate_model(name):
    model=models[name]; size=sizes[name]; pred=np.argmax(model.predict(make_ds(test_df,size,BATCH),verbose=1),axis=1); true=test_df.label.values
    report=classification_report(true,pred,output_dict=True,zero_division=0); cm=confusion_matrix(true,pred)
    folder=OUT/name; folder.mkdir(exist_ok=True)
    (folder/'classification_report.json').write_text(json.dumps(report,indent=2)); np.save(folder/'confusion_matrix.npy',cm)
    with open(folder/'history.json','w') as f: json.dump(histories[name],f,indent=2)
    plt.figure(figsize=(14,5)); plt.subplot(1,2,1); plt.plot(histories[name]['accuracy'],marker='o',label='train'); plt.plot(histories[name]['val_accuracy'],marker='o',label='validation'); plt.title(name+' accuracy'); plt.xlabel('Epoch'); plt.legend(); plt.grid(alpha=.2); plt.subplot(1,2,2); plt.plot(histories[name]['loss'],marker='o',label='train'); plt.plot(histories[name]['val_loss'],marker='o',label='validation'); plt.title(name+' loss'); plt.xlabel('Epoch'); plt.legend(); plt.grid(alpha=.2); plt.tight_layout(); plt.savefig(folder/'training_curves.png',dpi=160); plt.show()
    plt.figure(figsize=(9,8)); ConfusionMatrixDisplay(cm).plot(cmap='Blues',ax=plt.gca(),colorbar=False); plt.title(name+' official test confusion matrix'); plt.tight_layout(); plt.savefig(folder/'confusion_matrix.png',dpi=160); plt.show()
    print(name,{'accuracy':report['accuracy'],'macro_f1':report['macro avg']['f1-score'],'macro_precision':report['macro avg']['precision'],'macro_recall':report['macro avg']['recall']})
    return report
reports={name:evaluate_model(name) for name in models}
summary={'gpu':tf.config.get_visible_devices('GPU')[0].name,'models':{n:{'accuracy':r['accuracy'],'macro_f1':r['macro avg']['f1-score'],'epochs':len(histories[n]['loss'])} for n,r in reports.items()}}
(OUT/'results.json').write_text(json.dumps(summary,indent=2)); print(json.dumps(summary,indent=2))

In [ ]:
# Single-image prediction and a robust Grad-CAM visualization.
sample = str(test_df.iloc[0].path)
image = tf.keras.utils.load_img(sample, target_size=(128, 128))
arr = tf.keras.utils.img_to_array(image) / 255.0
x = tf.expand_dims(arr, 0)

prediction = models['mobilenet_v2'].predict(x, verbose=0)[0]
predicted = int(np.argmax(prediction))
print('sample:', sample, 'predicted_class:', predicted, 'confidence:', float(np.max(prediction)), 'true_class:', int(test_df.iloc[0].label))

# Compatible with Keras 2 and Keras 3 nested functional graphs
backbone = models['mobilenet_v2'].get_layer('mobilenet_backbone')
conv_layers = [l for l in backbone.layers if isinstance(l, (tf.keras.layers.Conv2D, tf.keras.layers.DepthwiseConv2D))]
last_conv = conv_layers[-1]

backbone_model = tf.keras.Model(backbone.input, [last_conv.output, backbone.output])

with tf.GradientTape() as tape:
    feat = x
    for layer in models['mobilenet_v2'].layers[1:]:
        if layer == backbone:
            conv_out, feat = backbone_model(feat, training=False)
        else:
            feat = layer(feat, training=False)
    score = feat[:, predicted]

grads = tape.gradient(score, conv_out)
weights = tf.reduce_mean(grads, axis=(1, 2))
heat = tf.maximum(tf.reduce_sum(conv_out * weights[:, None, None, :], axis=-1), 0)[0]
heat = heat / (tf.reduce_max(heat) + 1e-8)

plt.figure(figsize=(5, 5))
plt.imshow(arr)
plt.imshow(tf.image.resize(heat[..., None], [128, 128])[:, :, 0], cmap='jet', alpha=0.45)
plt.axis('off')
plt.title(f'Grad-CAM explanation (Class {predicted})')
plt.show()


In [ ]:
# Package all report-ready outputs and download directly to your computer.
bundle=shutil.make_archive('/content/gtsrb_colab_results','zip',OUT)
print('Download:',bundle)
from google.colab import files
files.download(bundle)

## Send back

After the download completes, send back `gtsrb_colab_results.zip` and the completed notebook (`File → Download → Download .ipynb`). I can then update the IEEE report, synopsis, presenter guide, presentation, and website with the Colab metrics and graphs.